In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from datasets import load_dataset
import torch.nn.functional as F

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model_ref = AutoModelForCausalLM.from_pretrained(model_name)

model = AutoModelForCausalLMWithValueHead.from_pretrained(model)
model_ref = AutoModelForCausalLMWithValueHead.from_pretrained(model_ref)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("imdb", split="train[:1000]")
dataset = dataset.filter(lambda x: len(x["text"]) > 200)

def build_dataset(tokenizer, dataset, max_length=256):
    def tokenize(sample):
        prompt = sample["text"][:200]
        sample["input_ids"] = tokenizer.encode(prompt, truncation=True, max_length=max_length)
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample
    return dataset.map(tokenize, batched=False)

dataset = build_dataset(tokenizer, dataset)

ppo_config = PPOConfig(
    batch_size=4,
    mini_batch_size=4,
    gradient_accumulation_steps=1,
    optimize_cuda_cache=True,
    learning_rate=1.41e-5,
    adam_eps=1e-5,
    max_grad_norm=0.5,
    log_with=None
)

ppo_trainer = PPOTrainer(ppo_config, model, model_ref, tokenizer)

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 0.9,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 50
}

def reward_fn(samples):
    device = model.pretrained_model.device
    sentiments = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english", device=device)
    results = sentiments(samples)
    rewards = [torch.tensor(1.0 if r["label"] == "POSITIVE" else -1.0) for r in results]
    return rewards

for epoch in range(3):
    for batch in torch.utils.data.DataLoader(dataset, batch_size=ppo_config.batch_size):
        query_tensors = [torch.tensor(t).long() for t in batch["input_ids"]]
        response_tensors = ppo_trainer.generate(query_tensors, return_prompt=False, **generation_kwargs)
        batch["response"] = tokenizer.batch_decode(response_tensors)

        rewards = reward_fn(batch["response"])
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)